# A certificate for the unroll depth

The [fixpoints notebook](fixpoints.ipynb) left one asymmetry: a *lossy* loop
knows its depth in advance through `unroll_depth`, but a lossless loop has no
guarantee at all — `fix` warns that nothing certifies its `max_steps`. The
stationary boson sampling bound closes that gap. For a feedback
interferometer $U$ on $M$ modes whose last $L$ wires are the loop, write
$U_{ll} = U[-L{:}, -L{:}]$ for the loop block and
$\beta_1 \ge \dots \ge \beta_L$ for the singular values of $U_{ll}^k$.
The claim is

$$\big\|\rho_{stat} - \rho^{(k)}\big\|_1
\;\le\; 4K(\bar q)\sum_{r=1}^L \arcsin^2\beta_r,
\qquad K(\bar q) = (\bar q+1)\Big[\sqrt{6\bar q(\bar q+1)} + \bar q\Big]$$

where $\rho^{(k)}$ is the loop state after $k$ round trips, $\rho_{stat}$
its fixpoint and $\bar q$ the largest photon number injected per mode. What
makes it strong is what the constant does *not* depend on: the depth, the
total photon number, the Fock cutoff, and any non-normality of $U_{ll}$ —
summing the $L$ singular directions instead of taking the worst one is what
kills the unboundable constant $C_U$ of $\|U_{ll}^k\| \le C_U\rho^k$.
`unroll_certificate` implements the resulting stopping rule: the smallest
$k$ whose bound falls below `tol`, exact in $O(kL^3)$, no eigenvalue and no
loss required. This notebook checks the claim against optyx's actual
unrolled loop states, then races the certificate against `unroll_depth`.

In [1]:
import numpy as np

from optyx import photonic
from optyx.channel import (
    Diagram, Discard, qmode, unroll_certificate)


def loop_over(unitary, injected=1):
    """The closed feedback loop of `unitary`: fresh photons in the first
    modes, every detector output discarded, the last mode fed back."""
    size = len(unitary)
    return (photonic.Create(*[injected] * (size - 1)) @ qmode
            >> photonic.Gate(np.asarray(unitary), size, size, "U")
            >> Discard(qmode ** (size - 1)) @ qmode
            ).feedback(mem=qmode, state=photonic.Create(0))


def loop_state(loop, steps):
    """The loop state after `steps` round trips, as a density matrix:
    unroll with the final memory left open and evaluate."""
    return loop.unroll_with_boundaries(
        steps - 1, effect=None).eval().density_matrix


def trace_norm(left, right):
    size = max(left.shape[0], right.shape[0])
    padded = [np.pad(x, [(0, size - x.shape[0])] * 2)
              for x in (left, right)]
    return np.abs(np.linalg.eigvalsh(padded[0] - padded[1])).sum()


K_ONE_PHOTON = 2 * (np.sqrt(12) + 1)

## Checking the claim

A two-mode loop whose block keeps an amplitude $\beta = 0.3$ per round
trip: the bound says the distance to the fixpoint decays as
$\arcsin^2(\beta^k) \approx \beta^{2k}$ — second order in the block, not
first, because photon-number conservation kills the odd orders. We unroll
the loop for real, take trace distances to a deep unrolling, and put the
bound next to them.

In [2]:
beta = 0.3
angle = np.arccos(beta)
unitary = np.array([[np.cos(angle), -np.sin(angle)],
                    [np.sin(angle), np.cos(angle)]])
loop = loop_over(unitary)
settled = loop_state(loop, 9)
print("  k   distance        bound      ratio")
for steps in range(1, 6):
    bound = 4 * K_ONE_PHOTON * np.arcsin(beta ** steps) ** 2
    distance = trace_norm(settled, loop_state(loop, steps))
    print(f"  {steps}   {distance:.3e}   {bound:.3e}   {bound/distance:6.1f}")

  k   distance        bound      ratio
  1   4.587e-01   3.315e+00      7.2
  2   2.627e-02   2.901e-01     11.0
  3   2.298e-03   2.604e-02     11.3


  4   2.063e-04   2.343e-03     11.4
  5   1.856e-05   2.109e-04     11.4


The bound dominates at every depth and tracks the true decay rate
$\beta^2 = 0.09$ per step exactly — the slack is a constant factor, not a
different exponent. That constant is the price of a bound that holds for
*any* reachable pair of states, uniformly in everything but $\bar q$.

## The unitarity floor

Light can only leave the loop through the $x = M - L$ external wires, so
$\|U_{ll}^k\| = 1$ exactly for every $k \le \lceil L/x\rceil - 1$,
whatever the unitary: no certificate can be shorter than
$\lceil L/x\rceil$. With three loop modes draining through one external
wire, the first two powers of the block are perfect isometries on some
direction — and the certificate respects the floor without being told.

In [3]:
def haar(size, seed):
    rng = np.random.default_rng(seed)
    z = rng.normal(size=(size, size)) + 1j * rng.normal(size=(size, size))
    q, r = np.linalg.qr(z)
    return q * (np.diag(r) / np.abs(np.diag(r)))


wide = haar(4, seed=7)
block = wide[-3:, -3:]
print("norms of the block powers:", [
    round(np.linalg.norm(np.linalg.matrix_power(block, k), 2), 6)
    for k in (1, 2, 3)])
print("certificate:", unroll_certificate(wide, loop_modes=3, tol=1e-3))

norms of the block powers: [1.0, 1.0, 0.963518]
certificate: 126


## The delay line reaches its fixpoint in one step

The embedding of ordinary boson sampling: $U = \mathrm{SWAP}\cdot
(W \oplus \mathrm{Id}_L)$ turns any single-pass circuit $W$ into a
feedback one whose loop is a pure delay. The loop block vanishes, so the
certificate is a single round trip — the reduction is exact and no mixing
time enters.

In [4]:
phase = np.exp(2j * np.pi * 0.3)
delay_line = np.array([[0, 1], [1, 0]]) @ np.diag([phase, 1])
print("loop block:", delay_line[-1:, -1:].item())
print("certificate:", unroll_certificate(delay_line, loop_modes=1))

loop block: 0j
certificate: 1


## Against `unroll_depth`

`unroll_depth` sees only the loss: $n^\star = \lceil\log(tol)/
\log(1-loss)\rceil$, and it refuses a lossless loop outright. The
certificate sees the geometry, and a loss just damps every singular value
by $\sqrt{1-loss}$ per round trip — the two agree up to the constant
$4K(\bar q)$ of the bound, and the certificate keeps working when the
loss is zero.

In [5]:
reflective = np.array([[.6, .8], [.8, -.6]])
loop = loop_over(reflective)
print("loss  unroll_depth  certificate")
for loss in (0, .5, .9):
    depth = (loop.unroll_depth(1e-6, loss) if loss else "refused")
    certificate = unroll_certificate(
        reflective, loop_modes=1, tol=1e-6, loss=loss)
    print(f" {loss:.1f}  {str(depth):>12}  {certificate:11}")

loss  unroll_depth  certificate
 0.0       refused           18
 0.5            20           11
 0.9             6            6


## The certificate closes `fix`'s warning

`fix(max_steps=...)` warns that no loss certifies its depth — the
certificate is exactly the number that discharges that warning by hand.
Contract at the certified depth and the result agrees with the
eigensolve to the tolerance the bound promised.

In [6]:
import warnings

certified = unroll_certificate(unitary, loop_modes=1, tol=1e-6)
measured = (photonic.Create(1) @ qmode
            >> photonic.Gate(unitary, 2, 2, "U")
            >> photonic.NumberResolvingMeasurement(1) @ qmode
            ).feedback(mem=qmode, state=photonic.Create(0))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    contracted = measured.fix(chi=None, max_steps=certified)
stationary = measured.eigen_fix()
disagreement = max(
    abs(contracted.prob_dist().get(k, 0) - v)
    for k, v in stationary.prob_dist().items())
print(f"certified depth: {certified}")
print(f"fix vs eigen_fix at that depth: {disagreement:.2e}")

certified depth: 8
fix vs eigen_fix at that depth: 3.94e-08


## Further reading

The bound and its proof are A. Lesage, *Stationary Boson Sampling* (2026),
internal notes, §2 — the second-order argument is §3.3–§3.7 of the full
manuscript. The setup and the eigensolve it certifies are Yu. A. Biriukov
and I. V. Dyakonov, [Simulation of boson sampling with optical
feedback](https://doi.org/10.48550/arXiv.2602.05566), arXiv:2602.05566
(2026); the complexity consequence — the looped architecture stays inside
boson sampling, Haar-typically and unconditionally under loss — is §3 of
the notes.